In [1]:
import pandas as pd

law_data = [
    {"law_id": 1, "section_code": "BNS 318", "act_name": "Bharatiya Nyaya Sanhita",
     "title": "Cheating", "plain_description": "Someone deceived you from the start to get your money or property"},
    {"law_id": 2, "section_code": "NI Act 138", "act_name": "Negotiable Instruments Act",
     "title": "Dishonour of cheque", "plain_description": "A cheque you received bounced due to insufficient funds"},
    {"law_id": 3, "section_code": "BNS 331", "act_name": "Bharatiya Nyaya Sanhita",
     "title": "Criminal trespass", "plain_description": "Someone entered or stayed on your property without permission"},
    {"law_id": 4, "section_code": "BNS 115", "act_name": "Bharatiya Nyaya Sanhita",
     "title": "Voluntarily causing hurt", "plain_description": "Someone physically hurt you"},
    {"law_id": 5, "section_code": "BNS 303", "act_name": "Bharatiya Nyaya Sanhita",
     "title": "Theft", "plain_description": "Someone took your property without your permission"},
    {"law_id": 6, "section_code": "BNS 296", "act_name": "Bharatiya Nyaya Sanhita",
     "title": "Public nuisance", "plain_description": "Someone's actions are disturbing your peace or property"},
    {"law_id": 7, "section_code": "BNS 356", "act_name": "Bharatiya Nyaya Sanhita",
     "title": "Defamation", "plain_description": "Someone made false statements that damaged your reputation"},
    {"law_id": 8, "section_code": "East Punjab Rent Restriction Act, Sec 13", "act_name": "Rent Restriction Act",
     "title": "Eviction grounds", "plain_description": "Your landlord is trying to evict you or dispute rent terms"},
]

law_df = pd.DataFrame(law_data)
print(f"Loaded {len(law_df)} law entries")
law_df.head()

Loaded 8 law entries


,law_id,section_code,act_name,title,plain_description
0,1,BNS 318,Bharatiya Nyaya Sanhita,Cheating,Someone deceived you from the start to get you...
1,2,NI Act 138,Negotiable Instruments Act,Dishonour of cheque,A cheque you received bounced due to insuffici...
2,3,BNS 331,Bharatiya Nyaya Sanhita,Criminal trespass,Someone entered or stayed on your property wit...
3,4,BNS 115,Bharatiya Nyaya Sanhita,Voluntarily causing hurt,Someone physically hurt you
4,5,BNS 303,Bharatiya Nyaya Sanhita,Theft,Someone took your property without your permis...


In [2]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

law_embeddings = model.encode(law_df["plain_description"].tolist(), normalize_embeddings=True)

print("Law embeddings shape:", law_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Law embeddings shape: (8, 384)


In [4]:
def match_situation(query_text, top_k=3, min_score=0.3):
    """
    Match a free-text situation to the most relevant law sections.
    Returns a list of matches with relevance scores, or empty list if no good match.
    """
    query_embedding = model.encode(query_text, normalize_embeddings=True)
    
    scores = util.cos_sim(query_embedding, law_embeddings)[0].numpy()
    
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    matches = []
    for idx in top_indices:
        score = float(scores[idx])
        if score < min_score:
            continue
        matches.append({
            "law_id": int(law_df.iloc[idx]["law_id"]),
            "section_code": law_df.iloc[idx]["section_code"],
            "act_name": law_df.iloc[idx]["act_name"],
            "plain_description": law_df.iloc[idx]["plain_description"],
            "relevance_score": round(score, 3),
        })
    
    return matches

In [5]:
test_queries = [
    "my neighbor built a wall on my land",
    "I lent someone money and they never paid me back",
    "someone slapped me during an argument",
    "my landlord is trying to kick me out",
    "someone is spreading lies about me in my community",
    "asdkjhaskjdh random text",
]

for q in test_queries:
    print(f"\nQuery: {q}")
    results = match_situation(q)
    if not results:
        print("  → No clear match found")
    else:
        for r in results:
            print(f"  → {r['section_code']} ({r['plain_description'][:50]}...) — score: {r['relevance_score']}")


Query: my neighbor built a wall on my land
  → BNS 303 (Someone took your property without your permission...) — score: 0.48
  → BNS 331 (Someone entered or stayed on your property without...) — score: 0.347
  → BNS 318 (Someone deceived you from the start to get your mo...) — score: 0.311

Query: I lent someone money and they never paid me back
  → BNS 318 (Someone deceived you from the start to get your mo...) — score: 0.439
  → NI Act 138 (A cheque you received bounced due to insufficient ...) — score: 0.382
  → BNS 303 (Someone took your property without your permission...) — score: 0.309

Query: someone slapped me during an argument
  → BNS 115 (Someone physically hurt you...) — score: 0.562
  → BNS 296 (Someone's actions are disturbing your peace or pro...) — score: 0.422
  → BNS 356 (Someone made false statements that damaged your re...) — score: 0.404

Query: my landlord is trying to kick me out
  → East Punjab Rent Restriction Act, Sec 13 (Your landlord is trying to evict you

In [6]:
for threshold in [0.2, 0.3, 0.4]:
    print(f"\n--- Threshold: {threshold} ---")
    results = match_situation("asdkjhaskjdh random text", min_score=threshold)
    print("Gibberish result:", results if results else "No match (correct)")


--- Threshold: 0.2 ---
Gibberish result: No match (correct)

--- Threshold: 0.3 ---
Gibberish result: No match (correct)

--- Threshold: 0.4 ---
Gibberish result: No match (correct)


## Situation Matching Prototype — Findings

- Built a reusable `match_situation()` function using `all-MiniLM-L6-v2`, tested against 
  8 placeholder law entries (will expand once Data Engineering delivers the full curated dataset).
- Precomputed law embeddings once at startup rather than per-query, matching the planned architecture.
- Tested 6 realistic queries — 5 real situations all correctly matched to sensible laws with 
  scores above [X]; gibberish input correctly returned no match at threshold 0.3.
- Function signature and output shape (law_id, section_code, relevance_score) already match 
  the API contract shared with Member 4 — minimal changes needed to wire into FastAPI.
- Next step: replace placeholder law_data with Data Engineering's real legal_sections.csv 
  once delivered, and expand test queries using the situation dataset I'll curate separately.